# PrecisionSkin V5 - Lesion Detector Pipeline (same structure as the classifier pipeline)

This notebook follows the same stages as `Pipeline V5 - With Facial Eczema.ipynb` (config -> manifest validation -> ablation -> multi-seed training ->
checkpoint audit -> best seed on VALIDATION only -> test set first and only touch -> confidence intervals -> error analysis -> leak-free re-evaluation ->
ONNX export + verification -> metadata + SHA-256 -> sanity assertions -> deployment summary), but for the **lesion detector** (YOLOv8, bounding boxes).

| Classifier pipeline | This pipeline |
|---|---|
| `CLASS_NAMES`, one 4-class model | `CONDITION` = `acne` / `hyperpigmentation` / `eczema`: one **specialist** per condition |
| `USE_AUXILIARY_NONFACIAL` ablation | `USE_NEGATIVES` ablation (healthy-skin background photos with no boxes) |
| split_manifest.csv | the YOLO dataset folders (train / val / test, plus the hand-boxed test) |
| macro-F1, accuracy, kappa | mAP@0.5, mAP@0.5-0.95, precision, recall, F1 at IoU 0.5 |
| Phase A / Phase B | one YOLO run per seed (the pretrained backbone is fine-tuned end to end) |
| Grad-CAM++ cells | not needed here (boxes are the localisation) |
| Fitzpatrick fairness matrix | skin-tone check is done with the separate MST model (not here) |

**Run order:** cells 1 -> 21 (Run All works; cell 14C scores the independent photos ONCE). Change only CELL 3 (settings). Outputs go to `PrecisionSkinV1/runs/lesion_<condition>_<run name>/`.
**Rule kept from the classifier pipeline:** the test photos are touched once, after the best seed has been chosen on validation. v1 (the detector in the app) is scored on the same photos for comparison.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 - GPU / Package Check
# ─────────────────────────────────────────────────────────────────────────────
import importlib.util, platform
need = ["ultralytics", "torch", "cv2", "numpy", "pandas", "yaml", "onnx", "onnxruntime", "matplotlib"]
missing = [m for m in need if importlib.util.find_spec(m) is None]
assert not missing, "Missing packages: %s  (in .venv_det: pip install ultralytics opencv-python-headless pandas pyyaml onnx onnxruntime matplotlib)" % missing
import torch, ultralytics
print("python", platform.python_version(), "| torch", torch.__version__, "| ultralytics", ultralytics.__version__)
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY (very slow)")

python 3.13.15 | torch 2.14.1+cu130 | ultralytics 8.4.172
CUDA: True | NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 - Project Root Verification
# ─────────────────────────────────────────────────────────────────────────────
import os
PROJECT_ROOT = "/home/bautista/PrecisionSkinV1"   # <-- update if needed
assert os.path.exists(PROJECT_ROOT), "Project folder not found: " + PROJECT_ROOT
print("Project root found:", PROJECT_ROOT)

Project root found: /home/bautista/PrecisionSkinV1


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 - Settings (the only cell you edit)
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, json, random, hashlib, shutil, time
from pathlib import Path
import numpy as np
import pandas as pd

SMOKE = False                  # True = 1-epoch plumbing test only (never use its numbers)

CONDITION = "acne"             # "acne" | "hyperpigmentation" | "eczema"   (one specialist detector per run)

# ---- ABLATION SWITCH (like USE_AUXILIARY_NONFACIAL in the classifier pipeline) ----
#   False: only the boxed photos of this condition
#   True : + healthy-skin background photos with NO boxes (about 10% of the training set) so the model learns that pores,
#          shadows and healthy texture are not lesions. Validation and test are never changed.
USE_NEGATIVES = False
NEGATIVES_FRACTION = 0.10
NORMAL_DIR = Path("/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Classes/Normal")

# ---- SECOND ABLATION SWITCH (acne only): the 60 training photos you re-reviewed in Fix_Acne_Missed_Spots (boxes 893 -> 1,430). Validation and test labels are never touched. ----
USE_FIXED_LABELS = False
FIX_DIR = Path("/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Acne_Boxing/Fix_Acne_Missed_Spots")

# ---- INDEPENDENT EVALUATION (acne): photos boxed blind, scored ONCE with the frozen protocol (thresholds 0.26 and 0.35, nothing else). Cell 14C. ----
INDEPENDENT_DIR = Path("/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Acne_Boxing/Independent_Final_Set")
FROZEN_THRESHOLDS = (0.26, 0.35)
ALLOW_RESCORE = False          # leave False: scoring the independent set twice would make it a development set

# ---- model / training (same settings as the acne runs so numbers stay comparable) ----
MODEL, IMGSZ, EPOCHS, PATIENCE, BATCH = "yolov8s.pt", 800, 100, 30, 8
SEEDS = [42, 7, 123]

# ---- per-condition data. Each dataset folder holds train/ val/ test/ (images + labels, YOLO format, class 0) ----
CONDITIONS = {
    "acne": dict(
        data=Path("/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Acne_Boxing/acne_dataset_v4"),
        test_dirs=["test"], primary_test="test", v1_class=0, v1_conf=0.20, app_threshold=0.26),
    "hyperpigmentation": dict(
        data=Path(PROJECT_ROOT) / "hyperpig_autolabel" / "hyperpig_only_v1" / "dataset",
        test_dirs=["hand_test", "test"], primary_test="hand_test", v1_class=1, v1_conf=0.25, app_threshold=0.175),
    "eczema": dict(
        data=Path(PROJECT_ROOT) / "eczema_dataset",
        test_dirs=["test"], primary_test="test", v1_class=2, v1_conf=0.25, app_threshold=0.25),
}
CFG = CONDITIONS[CONDITION]
V1_MODEL = Path(PROJECT_ROOT) / "detector_work" / "runs" / "detector_v1_yolov8n_seed42" / "weights" / "best.pt"   # the detector in the app (3 classes)
INSTALL_IN_APP = False         # True (cell 17) copies the ONNX + json into SkinDevApp/Model as the specialist for this condition
APP_MODEL_DIR = Path("/mnt/c/Users/Bautista/source/repos/SkinDevApp/SkinDevApp/Model")
HELPER_DIRS = [Path("/mnt/c/Users/Bautista/Downloads"), Path("/mnt/c/Users/Bautista/source/repos/SkinDevApp")]   # where hyperpig_autolabel.py (scoring code) is
BOOTSTRAPS = 1000
TRAIN_LIMIT = EVAL_LIMIT = 0   # smoke only

RUN_NAME = ("with_negatives" if USE_NEGATIVES else "boxed_only") + ("_fixedlabels" if USE_FIXED_LABELS else "")
if SMOKE:
    EPOCHS, IMGSZ, MODEL, SEEDS, BATCH, TRAIN_LIMIT, EVAL_LIMIT, BOOTSTRAPS = 1, 320, "yolov8n.pt", [42], 8, 60, 40, 50
    RUN_NAME += "_SMOKE"

RUN_ROOT = Path(PROJECT_ROOT) / "runs" / ("lesion_%s_%s" % (CONDITION, RUN_NAME))
MODEL_DIR, LOG_DIR, DATA_LOCAL, ERR_DIR, EXPORT_DIR = RUN_ROOT / "models", RUN_ROOT / "logs", RUN_ROOT / "data", RUN_ROOT / "error_review", RUN_ROOT / "export"
for d in (MODEL_DIR, LOG_DIR, ERR_DIR, EXPORT_DIR): d.mkdir(parents=True, exist_ok=True)

for d in HELPER_DIRS:
    if (d / "hyperpig_autolabel.py").exists():
        sys.path.insert(0, str(d)); break
else:
    raise FileNotFoundError("hyperpig_autolabel.py (the scoring code) not found in Downloads or the repo folder")
import hyperpig_autolabel as H

print("condition:", CONDITION, "| run:", RUN_NAME, "| model:", MODEL, "| imgsz", IMGSZ, "| seeds", SEEDS)
print("dataset  :", CFG["data"]); print("outputs  :", RUN_ROOT)

condition: acne | run: boxed_only | model: yolov8s.pt | imgsz 800 | seeds [42, 7, 123]
dataset  : /mnt/c/Users/Bautista/Downloads/PrecisionSkin/Acne_Boxing/acne_dataset_v4
outputs  : /home/bautista/PrecisionSkinV1/runs/lesion_acne_boxed_only


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 - Dataset / Manifest Validation
# ─────────────────────────────────────────────────────────────────────────────
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
def imgs_of(split_dir): return sorted(p for p in (Path(split_dir) / "images").glob("*") if p.suffix.lower() in IMG_EXT)
def read_label(p):
    out = []
    if Path(p).exists():
        for ln in Path(p).read_text().splitlines():
            t = ln.split()
            if len(t) >= 5: out.append(tuple(float(v) for v in t[1:5]))
    return out

src = CFG["data"]
assert src.exists(), "Dataset not found: %s (for eczema: build it first)" % src
splits = ["train", "val"] + CFG["test_dirs"]
rows, problems = [], []
for sp in splits:
    assert (src / sp / "images").exists(), "Missing split folder: %s" % (src / sp / "images")
    ims = imgs_of(src / sp); nb = 0; empty = 0
    for p in ims:
        lab = src / sp / "labels" / (p.stem + ".txt")
        bx = read_label(lab); nb += len(bx); empty += (len(bx) == 0)
        for t in bx:
            if not all(0.0 <= v <= 1.0 for v in t): problems.append((sp, p.name, "box outside 0..1"))
        if lab.exists():
            for ln in lab.read_text().splitlines():
                if ln.split() and int(float(ln.split()[0])) != 0: problems.append((sp, p.name, "class id is not 0"))
    rows.append(dict(split=sp, photos=len(ims), boxes=nb, boxes_per_photo=round(nb / max(1, len(ims)), 1), photos_without_boxes=empty))
manifest = pd.DataFrame(rows); print(manifest.to_string(index=False))
assert not problems, "label problems: %s" % problems[:5]
for sp in ["val"] + CFG["test_dirs"]: assert manifest[manifest.split == sp].photos.iloc[0] > 0, "split %s is empty" % sp
names = {sp: {p.stem for p in imgs_of(src / sp)} for sp in splits}
assert not (names["train"] & names["val"]), "a photo is in both train and val"
for t in CFG["test_dirs"]:
    assert not (names["train"] & names[t]) and not (names["val"] & names[t]), "a photo is in train/val AND %s" % t
dataset_fingerprint = hashlib.sha256("\n".join(sorted("%s/%s" % (sp, p.name) for sp in splits for p in imgs_of(src / sp))).encode()).hexdigest()
manifest.to_csv(LOG_DIR / "dataset_manifest.csv", index=False)
print("\nfile-name leak check passed | dataset fingerprint:", dataset_fingerprint[:16])
print("primary test set:", CFG["primary_test"], "| other test sets:", [t for t in CFG["test_dirs"] if t != CFG["primary_test"]])

split  photos  boxes  boxes_per_photo  photos_without_boxes
train    1145  11621             10.1                     0
  val     154   1562             10.1                     0
 test     167   1369              8.2                     0

file-name leak check passed | dataset fingerprint: ae51483a7b72c113
primary test set: test | other test sets: []


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 - Look-alike Leak Check (test vs train/val, 8 rotations/flips)
# ─────────────────────────────────────────────────────────────────────────────
# Same idea as the classifier pipeline's leak checks: public datasets contain augmented copies of the same photo.
def thumbs(split):
    out = []
    for p in imgs_of(src / split):
        t = H.thumb(p)
        if t is not None: out.append((p.name, t))
    return out
tr_val = thumbs("train") + thumbs("val")
Oz = np.stack([t for _, t in tr_val])
leaks = []
for tsplit in CFG["test_dirs"]:
    T = thumbs(tsplit)
    best = H.best_corr(np.stack([t for _, t in T]), Oz)
    for i, (nm, _) in enumerate(T):
        j = int(best[i].argmax())
        if best[i, j] >= 0.90: leaks.append(dict(test_split=tsplit, test_photo=nm, similar_train_val_photo=tr_val[j][0], correlation=round(float(best[i, j]), 3)))
leak_df = pd.DataFrame(leaks, columns=["test_split", "test_photo", "similar_train_val_photo", "correlation"])
leak_df.to_csv(LOG_DIR / "leak_check.csv", index=False)
print("test photos with a look-alike in train/val (corr >= 0.90):", len(leak_df))
print(leak_df.groupby("test_split").size().to_string() if len(leak_df) else "none")
print("These are reported in cell 14 as a leak-free re-evaluation; nothing is deleted here.")

test photos with a look-alike in train/val (corr >= 0.90): 2
test_split
test    2
These are reported in cell 14 as a leak-free re-evaluation; nothing is deleted here.


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 - Build the local training data (+ optional negatives = the ablation)
# ─────────────────────────────────────────────────────────────────────────────
def copy_split(sp, limit=0):
    (DATA_LOCAL / sp / "images").mkdir(parents=True, exist_ok=True); (DATA_LOCAL / sp / "labels").mkdir(parents=True, exist_ok=True)
    ims = imgs_of(src / sp)
    if limit: ims = ims[:limit]
    for p in ims:
        shutil.copy2(p, DATA_LOCAL / sp / "images" / p.name)
        lab = src / sp / "labels" / (p.stem + ".txt")
        shutil.copy2(lab, DATA_LOCAL / sp / "labels" / lab.name) if lab.exists() else (DATA_LOCAL / sp / "labels" / lab.name).write_text("")
if not (DATA_LOCAL / "READY").exists():
    shutil.rmtree(DATA_LOCAL, ignore_errors=True)
    for sp in splits: copy_split(sp, (TRAIN_LIMIT if sp == "train" else EVAL_LIMIT) if SMOKE else 0)
    n_fixed = 0
    if USE_FIXED_LABELS and CONDITION == "acne":                       # your re-reviewed labels replace the old ones in the LOCAL copy of the training photos (originals untouched)
        fix_labels = FIX_DIR / "Elaiza__acne_fix" / "labels"; ref = json.load(open(FIX_DIR / "_drafts" / "meta.json"))["drafted_at"]
        for f in fix_labels.glob("*.txt"):
            if f.stat().st_mtime > ref + 1 and (DATA_LOCAL / "train" / "labels" / f.name).exists():      # saved by you = reviewed; training photos only
                shutil.copy2(f, DATA_LOCAL / "train" / "labels" / f.name); n_fixed += 1
        print("relabelled training photos applied:", n_fixed)
    n_neg = 0
    if USE_NEGATIVES:
        import cv2
        def junk(path, min_side=400):
            im = cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_COLOR)
            if im is None: return "unreadable"
            h, w = im.shape[:2]
            if min(h, w) < min_side: return "small"
            g = cv2.cvtColor(cv2.resize(im, (128, 128), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY).astype(np.float32)
            if (g <= 8).mean() > 0.02: return "black corners (rotated crop)"
            for strip, ax in ((g[:, :12], 1), (g[:, -12:], 1), (g[:12, :], 0), (g[-12:, :], 0)):
                if np.abs(np.diff(strip, axis=ax)).mean() < 0.6: return "smeared border"
            return None
        from ultralytics import YOLO
        probe = YOLO(str(V1_MODEL)) if V1_MODEL.exists() else None
        want = int(round(NEGATIVES_FRACTION * len(imgs_of(DATA_LOCAL / "train"))))
        cand = sorted(p for p in NORMAL_DIR.rglob("*") if p.suffix.lower() in IMG_EXT); random.Random(5).shuffle(cand)
        why, kept = {}, []
        for p in cand:
            r = junk(p)
            if r: why[r] = why.get(r, 0) + 1; continue
            if probe is not None:                                    # a "healthy" photo on which the app detector fires confidently may really show lesions
                det = probe.predict(str(p), imgsz=640, conf=0.50, verbose=False, device=0 if torch.cuda.is_available() else "cpu")[0]
                if det.boxes is not None and len(det.boxes): why["lesion-like (v1 conf>=0.5)"] = why.get("lesion-like (v1 conf>=0.5)", 0) + 1; continue
            kept.append(p)
            if len(kept) >= want: break
        for i, p in enumerate(kept):
            shutil.copy2(p, DATA_LOCAL / "train/images" / ("neg_%03d%s" % (i, p.suffix.lower()))); (DATA_LOCAL / "train/labels" / ("neg_%03d.txt" % i)).write_text("")
        n_neg = len(kept); print("negatives added: %d (wanted %d) | rejected: %s" % (n_neg, want, why))
    (DATA_LOCAL / "READY").write_text(json.dumps(dict(negatives=n_neg, fixed_labels=n_fixed, smoke=SMOKE)))
info = json.loads((DATA_LOCAL / "READY").read_text())
print({sp: len(imgs_of(DATA_LOCAL / sp)) for sp in splits}, "| negatives in train:", info["negatives"])
assert (info.get("fixed_labels", 0) > 0) == (USE_FIXED_LABELS and CONDITION == "acne") or SMOKE, "this data folder was built with a different USE_FIXED_LABELS: use another RUN_NAME / delete the data folder"
assert info["negatives"] == 0 or USE_NEGATIVES, "this data folder contains negatives but USE_NEGATIVES is False: use a different RUN_NAME / delete the data folder"

{'train': 1145, 'val': 154, 'test': 167} | negatives in train: 0


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 - Training (all seeds, resumable: re-run this cell after a crash)
# ─────────────────────────────────────────────────────────────────────────────
import yaml
from ultralytics import YOLO
train_list = sorted(imgs_of(DATA_LOCAL / "train"))
seed_runs = {}
for seed in SEEDS:
    name = "seed_%d" % seed; run = MODEL_DIR / name; best, last, done = run / "weights" / "best.pt", run / "weights" / "last.pt", run / "DONE"
    lst = LOG_DIR / ("train_%s.txt" % name); lst.write_text("\n".join(str(p) for p in train_list) + "\n")
    yml = LOG_DIR / ("data_%s.yaml" % name)
    yaml.safe_dump({"path": str(DATA_LOCAL), "train": str(lst), "val": "val/images", "test": CFG["primary_test"] + "/images", "names": {0: CONDITION}}, open(yml, "w"))
    seed_runs[seed] = dict(name=name, best=best, last=last, run=run)
    if done.exists(): print("[%s] already trained -> skipped" % name); continue
    print("=" * 30, name, "| training photos:", len(train_list))
    if last.exists(): YOLO(str(last)).train(resume=True)
    else:
        YOLO(MODEL).train(data=str(yml), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, device=0 if torch.cuda.is_available() else "cpu", workers=2,
                          seed=seed, cos_lr=True, close_mosaic=10, project=str(MODEL_DIR), name=name, exist_ok=True,
                          fliplr=0.5, flipud=0.0, degrees=10, scale=0.5, hsv_h=0.015, hsv_s=0.5, hsv_v=0.35)
    assert best.exists(), "training produced no best.pt for " + name
    done.write_text("ok")
print("\ntrained seeds:", list(seed_runs))

============================== seed_42 | training photos: 1145
New https://pypi.org/project/ultralytics/8.4.175 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/bautista/PrecisionSkinV1/runs/lesion_acne_boxed_only/logs/data_seed_42.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.35, imgs

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 - Post-training Checkpoint Audit
# ─────────────────────────────────────────────────────────────────────────────
aud = []
for seed, r in seed_runs.items():
    rc = r["run"] / "results.csv"
    df = pd.read_csv(rc); df.columns = [c.strip() for c in df.columns]
    be = int(df["metrics/mAP50(B)"].idxmax()) + 1
    aud.append(dict(seed=seed, best_pt=r["best"].exists(), last_pt=r["last"].exists(), done=(r["run"] / "DONE").exists(), epochs_run=len(df), best_epoch=be,
                    stopped_early=len(df) < EPOCHS, best_val_mAP50=round(float(df["metrics/mAP50(B)"].max()), 3)))
audit = pd.DataFrame(aud); audit.to_csv(LOG_DIR / "checkpoint_audit.csv", index=False); print(audit.to_string(index=False))
assert audit.best_pt.all() and audit.done.all(), "a seed has no best.pt / DONE marker"

 seed  best_pt  last_pt  done  epochs_run  best_epoch  stopped_early  best_val_mAP50
   42     True     True  True         100          78          False           0.432
    7     True     True  True          79          50           True           0.424
  123     True     True  True          85          32           True           0.430


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 - Best Seed Selection (VALIDATION only)
# ─────────────────────────────────────────────────────────────────────────────
val_rows = []
for seed, r in seed_runs.items():
    sv = H.score_model(r["best"], IMGSZ, DATA_LOCAL / "val")
    conf = H.best_f1_conf(sv); p, rc, f = H.at_conf(sv, conf)
    val_rows.append(dict(seed=seed, val_mAP50=sv["AP50"], val_mAP50_95=sv["AP50_95"], val_conf=conf, val_precision=p, val_recall=rc, val_F1=f)); print("seed %d: val mAP50 %.3f | F1 %.3f at conf %.3f" % (seed, sv["AP50"], f, conf))
val_df = pd.DataFrame(val_rows); val_df.to_csv(LOG_DIR / "val_seed_selection.csv", index=False)
best_seed = int(val_df.loc[val_df["val_mAP50"].idxmax(), "seed"]); seed_conf = {int(r.seed): float(r.val_conf) for r in val_df.itertuples()}
print("\nBest seed selected using VALIDATION only:", best_seed, "| the test set has not been used for model selection.")

seed 42: val mAP50 0.425 | F1 0.494 at conf 0.250
seed 7: val mAP50 0.429 | F1 0.479 at conf 0.225
seed 123: val mAP50 0.440 | F1 0.494 at conf 0.250

Best seed selected using VALIDATION only: 123 | the test set has not been used for model selection.


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 - Test Set Evaluation (first and only test touch) + v1 comparison
# ─────────────────────────────────────────────────────────────────────────────
TEST = DATA_LOCAL / CFG["primary_test"]
test_rows = []
for seed, r in seed_runs.items():
    st = H.score_model(r["best"], IMGSZ, TEST); c = seed_conf[seed]; p, rc, f = H.at_conf(st, c)
    test_rows.append(dict(seed=seed, conf=round(c, 3), test_mAP50=st["AP50"], test_mAP50_95=st["AP50_95"], precision=p, recall=rc, F1=f)); print("seed %d: test mAP50 %.3f | P %.2f R %.2f F1 %.2f" % (seed, st["AP50"], p, rc, f))
test_df = pd.DataFrame(test_rows); test_df.to_csv(LOG_DIR / ("seed_results_%s.csv" % RUN_NAME), index=False)
print("\nmean +/- std over %d seeds:" % len(test_df))
for col in ("test_mAP50", "test_mAP50_95", "precision", "recall", "F1"):
    print("  %-14s %.3f +/- %.3f" % (col, test_df[col].mean(), test_df[col].std(ddof=1) if len(test_df) > 1 else 0.0))
v1 = None
if V1_MODEL.exists():
    sv1 = H.score_model(V1_MODEL, 640, TEST, CFG["v1_class"]); p, rc, f = H.at_conf(sv1, CFG["v1_conf"])
    v1 = dict(test_mAP50=sv1["AP50"], test_mAP50_95=sv1["AP50_95"], precision=p, recall=rc, F1=f)
    print("\nv1 (in the app, class %d, conf %.2f): mAP50 %.3f | P %.2f R %.2f F1 %.2f | new minus v1 = %+.3f mAP50" % (CFG["v1_class"], CFG["v1_conf"], v1["test_mAP50"], p, rc, f, test_df.test_mAP50.mean() - v1["test_mAP50"]))
print("\nNoise note: differences below ~0.02-0.03 mAP50 are inside the run-to-run and test-set noise.")

seed 42: test mAP50 0.412 | P 0.43 R 0.54 F1 0.48
seed 7: test mAP50 0.408 | P 0.41 R 0.55 F1 0.47
seed 123: test mAP50 0.419 | P 0.44 R 0.54 F1 0.49

mean +/- std over 3 seeds:
  test_mAP50     0.413 +/- 0.006
  test_mAP50_95  0.123 +/- 0.003
  precision      0.427 +/- 0.015
  recall         0.543 +/- 0.006
  F1             0.478 +/- 0.008

v1 (in the app, class 0, conf 0.20): mAP50 0.220 | P 0.36 R 0.23 F1 0.28 | new minus v1 = +0.193 mAP50

Noise note: differences below ~0.02-0.03 mAP50 are inside the run-to-run and test-set noise.


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 - Confidence-threshold sweep (precision vs recall) on the TEST photos, 3-seed average
# ─────────────────────────────────────────────────────────────────────────────
res_all = [H.score_model(r["best"], IMGSZ, TEST) for r in seed_runs.values()]
sweep = []
for c in (0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60):
    pr = [H.at_conf(x, c) for x in res_all]
    sweep.append(dict(conf=c, precision=np.mean([a[0] for a in pr]), recall=np.mean([a[1] for a in pr]), F1=np.mean([a[2] for a in pr]),
                      boxes_per_photo=np.mean([(x["conf"] >= c).sum() for x in res_all]) / res_all[0]["n_photos"]))
sweep_df = pd.DataFrame(sweep).round(3); sweep_df.to_csv(LOG_DIR / "threshold_sweep.csv", index=False)
print(sweep_df.to_string(index=False)); print("true boxes per photo: %.1f" % (res_all[0]["n_gt"] / res_all[0]["n_photos"]))
deploy_conf = float(np.round(np.mean(list(seed_conf.values())), 3))
print("\nconfidence chosen on VALIDATION (mean of the seeds): %.3f | app default for this condition: %.3f (pick by how many false boxes the screen should tolerate)" % (deploy_conf, CFG["app_threshold"]))

 conf  precision  recall    F1  boxes_per_photo
 0.15      0.374   0.601 0.461           13.180
 0.20      0.404   0.565 0.471           11.465
 0.25      0.434   0.537 0.480           10.146
 0.30      0.461   0.507 0.483            9.020
 0.35      0.484   0.478 0.481            8.108
 0.40      0.504   0.439 0.469            7.140
 0.50      0.567   0.350 0.430            5.098
 0.60      0.615   0.204 0.300            2.762
true boxes per photo: 8.2

confidence chosen on VALIDATION (mean of the seeds): 0.242 | app default for this condition: 0.260 (pick by how many false boxes the screen should tolerate)


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 - Bootstrap Confidence Intervals (resampling test PHOTOS)
# ─────────────────────────────────────────────────────────────────────────────
def per_photo(weights, imgsz, split_dir, class_id=None):
    m = YOLO(str(weights)); out = []; ims = imgs_of(split_dir)
    for s in range(0, len(ims), 16):
        chunk = ims[s:s + 16]
        res = m.predict([str(p) for p in chunk], imgsz=imgsz, conf=0.001, iou=0.6, max_det=300, verbose=False, device=0 if torch.cuda.is_available() else "cpu")
        for p, r in zip(chunk, res):
            h, w = r.orig_shape
            g = np.array([[(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h] for cx, cy, bw, bh in read_label(Path(split_dir) / "labels" / (p.stem + ".txt"))], np.float32).reshape(-1, 4)
            if r.boxes is None or len(r.boxes) == 0: out.append((np.zeros(0), np.zeros(0), len(g))); continue
            b = r.boxes.xyxy.cpu().numpy(); c = r.boxes.conf.cpu().numpy()
            if class_id is not None:
                keep = r.boxes.cls.cpu().numpy().astype(int) == class_id; b, c = b[keep], c[keep]
            o = np.argsort(-c); b, c = b[o], c[o]; iou = H._iou_matrix(b, g); used = set(); flag = np.zeros(len(b))
            for i in range(len(b)):
                cand = [(iou[i, j], j) for j in range(iou.shape[1]) if j not in used and iou[i, j] >= 0.5] if iou.shape[1] else []
                if cand: j = max(cand)[1]; used.add(j); flag[i] = 1
            out.append((c, flag, len(g)))
    return out
def ap_at(photos, idx):
    c = np.concatenate([photos[i][0] for i in idx]); t = np.concatenate([photos[i][1] for i in idx]); n = sum(photos[i][2] for i in idx)
    return H._ap(c, t, n)
pp = {seed: per_photo(r["best"], IMGSZ, TEST) for seed, r in seed_runs.items()}
n = len(next(iter(pp.values()))); rng = np.random.default_rng(0); boots = []
pp_v1 = per_photo(V1_MODEL, 640, TEST, CFG["v1_class"]) if v1 is not None else None
for _ in range(BOOTSTRAPS):
    idx = rng.integers(0, n, n); new = np.mean([ap_at(pp[s], idx) for s in pp]); old = ap_at(pp_v1, idx) if pp_v1 else float("nan"); boots.append((new, old, new - old))
b = np.array(boots); lo, hi = np.percentile(b[:, 0], [2.5, 97.5])
print("mean mAP@0.5 of the new seeds: %.3f  (95%% range %.3f to %.3f, %d resamples of %d test photos)" % (test_df.test_mAP50.mean(), lo, hi, BOOTSTRAPS, n))
if pp_v1:
    dl, dh = np.nanpercentile(b[:, 2], [2.5, 97.5]); print("gain over v1: %+.3f  (95%% range %+.3f to %+.3f) -> %s" % (np.nanmean(b[:, 2]), dl, dh, "real, not noise" if dl > 0 else "not proven"))
json.dump(dict(mean_mAP50=float(test_df.test_mAP50.mean()), ci95=[float(lo), float(hi)], n_test_photos=n), open(LOG_DIR / "bootstrap.json", "w"))

mean mAP@0.5 of the new seeds: 0.413  (95% range 0.369 to 0.463, 1000 resamples of 167 test photos)
gain over v1: +0.193  (95% range +0.147 to +0.238) -> real, not noise


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 - Error Analysis (recall by box size, false boxes, review sheet)
# ─────────────────────────────────────────────────────────────────────────────
import cv2
def iou1(a, b):
    ax0, ay0, ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2; bx0, by0, bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    iw, ih = max(0, min(ax1, bx1) - max(ax0, bx0)), max(0, min(ay1, by1) - max(ay0, by0)); i = iw * ih
    return i / ((ax1 - ax0) * (ay1 - ay0) + (bx1 - bx0) * (by1 - by0) - i + 1e-9)
bins = [(0, 0.02, "tiny (<2% of photo side)"), (0.02, 0.04, "small (2-4%)"), (0.04, 0.08, "medium (4-8%)"), (0.08, 9, "large (>8%)")]
mbest = YOLO(str(seed_runs[best_seed]["best"])); conf_b = seed_conf[best_seed]
gt_n, hit, fp_total, rows_fp = [0] * 4, [0] * 4, 0, []
for p in imgs_of(TEST):
    gt = read_label(TEST / "labels" / (p.stem + ".txt"))
    r = mbest.predict(str(p), imgsz=IMGSZ, conf=conf_b, iou=0.5, verbose=False, device=0 if torch.cuda.is_available() else "cpu")[0]
    pr = [tuple(map(float, x)) for x in r.boxes.xywhn.cpu().numpy()] if len(r.boxes) else []; used = set()
    for g in gt:
        k = [i for i, (lo_, hi_, _) in enumerate(bins) if lo_ <= (g[2] * g[3]) ** 0.5 < hi_][0]; gt_n[k] += 1
        best_i, bj = 0, -1
        for j, q in enumerate(pr):
            if j not in used and iou1(g, q) > best_i: best_i, bj = iou1(g, q), j
        if best_i >= 0.5: used.add(bj); hit[k] += 1
    fps = [q for j, q in enumerate(pr) if j not in used]; fp_total += len(fps); rows_fp.append((len(fps), p, gt, fps))
print("recall by true box size (best seed, conf %.3f):" % conf_b)
for k, (_, _, nm) in enumerate(bins): print("  %-26s found %4d of %4d (%.0f%%)" % (nm, hit[k], gt_n[k], 100 * hit[k] / max(1, gt_n[k])))
print("false boxes (no match to your box at IoU 0.5): %d over %d photos" % (fp_total, len(rows_fp)))
rows_fp.sort(key=lambda t: -t[0]); step = max(1, len(rows_fp) // 24)
for k, (nfp, p, gt, fps) in enumerate(rows_fp[::step][:24]):
    im = cv2.imdecode(np.fromfile(str(p), np.uint8), cv2.IMREAD_COLOR); H_, W_ = im.shape[:2]; th = max(2, W_ // 300)
    for bxs, col in ((gt, (0, 255, 0)), (fps, (0, 0, 255))):
        for (cx, cy, w, h) in bxs: cv2.rectangle(im, (int((cx - w / 2) * W_), int((cy - h / 2) * H_)), (int((cx + w / 2) * W_), int((cy + h / 2) * H_)), col, th)
    cv2.imencode(".jpg", im)[1].tofile(str(ERR_DIR / ("%02d_%s.jpg" % (k + 1, p.stem))))
print("review sheet (green = your boxes, red = model boxes with no match):", ERR_DIR)

recall by true box size (best seed, conf 0.250):
  tiny (<2% of photo side)   found    1 of   20 (5%)
  small (2-4%)               found  103 of  251 (41%)
  medium (4-8%)              found  387 of  731 (53%)
  large (>8%)                found  238 of  367 (65%)
false boxes (no match to your box at IoU 0.5): 926 over 167 photos
review sheet (green = your boxes, red = model boxes with no match): /home/bautista/PrecisionSkinV1/runs/lesion_acne_boxed_only/error_review


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 - Leak-free Re-evaluation (test photos with a look-alike in train/val removed)
# ─────────────────────────────────────────────────────────────────────────────
bad = set(leak_df[leak_df.test_split == CFG["primary_test"]].test_photo)
if not bad:
    print("no look-alike test photos -> nothing to remove; the test result above is already leak-free")
    leakfree = dict(n_removed=0, test_mAP50=float(test_df.test_mAP50.mean()))
else:
    SUB = RUN_ROOT / "test_leakfree"; shutil.rmtree(SUB, ignore_errors=True)
    for p in imgs_of(TEST):
        if p.name in bad: continue
        (SUB / "images").mkdir(parents=True, exist_ok=True); (SUB / "labels").mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, SUB / "images" / p.name); shutil.copy2(TEST / "labels" / (p.stem + ".txt"), SUB / "labels" / (p.stem + ".txt"))
    lf = [H.score_model(r["best"], IMGSZ, SUB)["AP50"] for r in seed_runs.values()]
    leakfree = dict(n_removed=len(bad), test_mAP50=float(np.mean(lf)))
    print("removed %d look-alike test photos | mean mAP@0.5 on the rest: %.3f (all photos: %.3f)" % (len(bad), leakfree["test_mAP50"], test_df.test_mAP50.mean()))
json.dump(leakfree, open(LOG_DIR / "leakfree.json", "w"))

removed 2 look-alike test photos | mean mAP@0.5 on the rest: 0.418 (all photos: 0.413)


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14B - Usage-level reliability on the development test (photo / count / zone level)
# ─────────────────────────────────────────────────────────────────────────────
# mAP at strict IoU punishes a few pixels of box edge on a 30 px spot. These ask what the app is used for (screening aid). Acne only; healthy photos are NOT verified.
if CONDITION == "acne":
    def zones(bx):
        z = np.zeros((3, 3), bool)
        for cx, cy, w, h in bx: z[min(2, int(cy * 3)), min(2, int(cx * 3))] = True
        return z
    def auc_of(pos, neg):
        s = np.concatenate([pos, neg]); y = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))]); o = np.argsort(s); r = np.empty(len(s)); r[o] = np.arange(1, len(s) + 1)
        for v in np.unique(s): mm = s == v; r[mm] = r[mm].mean()
        return (r[y == 1].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))
    import cv2
    healthy = []
    for p in sorted(q for q in NORMAL_DIR.rglob("*") if q.suffix.lower() in IMG_EXT)[::7]:
        im = cv2.imdecode(np.fromfile(str(p), np.uint8), cv2.IMREAD_COLOR)
        if im is not None and min(im.shape[:2]) >= 400: healthy.append(p)
        if len(healthy) >= 200: break
    pos = imgs_of(TEST); rows14 = []
    for seed, r in seed_runs.items():
        mm = YOLO(str(r["best"]))
        def run(p):
            try: o = mm.predict(str(p), imgsz=IMGSZ, conf=0.05, iou=0.5, verbose=False, device=0 if torch.cuda.is_available() else "cpu")[0]
            except Exception: return None
            return (o.boxes.xywhn.cpu().numpy(), o.boxes.conf.cpu().numpy()) if len(o.boxes) else (np.zeros((0, 4)), np.zeros(0))
        P_ = [(p, run(p)) for p in pos]; P_ = [(p, x) for p, x in P_ if x is not None]; N_ = [x for x in (run(p) for p in healthy) if x is not None]
        for thr in FROZEN_THRESHOLDS:
            npred = np.array([int((c >= thr).sum()) for _, (b, c) in P_]); ngt = np.array([len(read_label(TEST / "labels" / (p.stem + ".txt"))) for p, _ in P_])
            tp = fp = fn = 0
            for p, (b, c) in P_:
                zp = zones([x for x, cc in zip(b, c) if cc >= thr]); zg = zones(read_label(TEST / "labels" / (p.stem + ".txt"))); tp += int((zp & zg).sum()); fp += int((zp & ~zg).sum()); fn += int((~zp & zg).sum())
            zP, zR = tp / max(1, tp + fp), tp / max(1, tp + fn)
            rows14.append(dict(seed=seed, thr=thr, AUC=auc_of(np.array([c.max() if len(c) else 0.0 for _, (b, c) in P_]), np.array([c.max() if len(c) else 0.0 for b, c in N_])),
                               sensitivity=float(np.mean([(c >= thr).any() if len(c) else False for _, (b, c) in P_])), specificity=float(np.mean([not (c >= thr).any() if len(c) else True for b, c in N_])),
                               count_spearman=float(np.corrcoef(np.argsort(np.argsort(npred)), np.argsort(np.argsort(ngt)))[0, 1]), count_MAE=float(np.abs(npred - ngt).mean()), zone_P=zP, zone_R=zR, zone_F1=2 * zP * zR / max(1e-9, zP + zR)))
    u = pd.DataFrame(rows14); u.to_csv(LOG_DIR / "usage_level_metrics.csv", index=False)
    print(u.groupby("thr").agg(["mean", "std"]).round(3).T.to_string())
    print("healthy photos used: %d (Normal class, NOT individually verified; a few may show faint acne, which lowers specificity)" % len(healthy))
else:
    print("usage-level metrics are defined for acne only")

thr                    0.26    0.35
seed           mean  57.333  57.333
               std   59.501  59.501
AUC            mean   0.962   0.962
               std    0.003   0.003
sensitivity    mean   0.970   0.950
               std    0.012   0.009
specificity    mean   0.685   0.825
               std    0.039   0.035
count_spearman mean   0.733   0.704
               std    0.011   0.023
count_MAE      mean   4.212   3.936
               std    0.048   0.081
zone_P         mean   0.789   0.827
               std    0.004   0.007
zone_R         mean   0.854   0.794
               std    0.005   0.031
zone_F1        mean   0.820   0.810
               std    0.003   0.013
healthy photos used: 200 (Normal class, NOT individually verified; a few may show faint acne, which lowers specificity)


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14C - INDEPENDENT evaluation (blind-boxed photos, scored ONCE, frozen protocol)
# ─────────────────────────────────────────────────────────────────────────────
# Frozen BEFORE these photos were boxed: model = the best seed chosen on VALIDATION (cell 9), thresholds 0.26 and 0.35 only. The photos were boxed from scratch, no drafts, X = no acne.
indep_result = None
if CONDITION == "acne" and INDEPENDENT_DIR.exists():
    exp = INDEPENDENT_DIR / "boxed_independent"; lock = INDEPENDENT_DIR / "SCORED.json"
    boxed = imgs_of(exp); xfile = exp / "no_lesion_marked_X.txt"
    x_names = [l.strip() for l in xfile.read_text().splitlines() if l.strip()] if xfile.exists() else []
    print("independent set: %d photos with boxes | %d marked X (no acne)" % (len(boxed), len(x_names)))
    if lock.exists() and not ALLOW_RESCORE and not SMOKE:
        print("ALREADY SCORED once - not scoring again (it would become a development set). Stored result:\n", lock.read_text())
        indep_result = json.loads(lock.read_text())
    else:
        w = seed_runs[best_seed]["best"]; ind = {}
        sv = H.score_model(w, IMGSZ, exp); ind["mAP50"] = float(sv["AP50"]); ind["mAP50_95"] = float(sv["AP50_95"]); ind["n_photos"] = len(boxed); ind["n_boxes"] = int(sv["n_gt"])
        ppi = per_photo(w, IMGSZ, exp); nI = len(ppi); rng_i = np.random.default_rng(0)
        def prf(idx, thr):
            c = np.concatenate([ppi[i][0] for i in idx]); t_ = np.concatenate([ppi[i][1] for i in idx]); n_gt = sum(ppi[i][2] for i in idx); m_ = c >= thr
            p_ = t_[m_].sum() / max(1, m_.sum()); r_ = t_[m_].sum() / max(1, n_gt); return p_, r_, 2 * p_ * r_ / max(1e-9, p_ + r_)
        ind["bootstrap_mAP50_95pct_range"] = [float(x) for x in np.percentile([ap_at(ppi, rng_i.integers(0, nI, nI)) for _ in range(BOOTSTRAPS)], [2.5, 97.5])]
        ind["at_threshold"] = {}
        for thr in FROZEN_THRESHOLDS:
            p_, r_, f_ = prf(range(nI), thr); bs = np.array([prf(rng_i.integers(0, nI, nI), thr) for _ in range(BOOTSTRAPS)])
            ind["at_threshold"][str(thr)] = dict(precision=float(p_), recall=float(r_), F1=float(f_), F1_95pct_range=[float(x) for x in np.percentile(bs[:, 2], [2.5, 97.5])],
                                                 photo_sensitivity=float(np.mean([(c >= thr).any() if len(c) else False for c, _, _ in ppi])))
        if x_names:
            mx_ = YOLO(str(w)); xs = [imgs_of(INDEPENDENT_DIR / "Elaiza__acne_independent")]
            xp = [INDEPENDENT_DIR / "Elaiza__acne_independent" / "images" / n for n in x_names if (INDEPENDENT_DIR / "Elaiza__acne_independent" / "images" / n).exists()]
            ind["X_photos"] = {str(thr): dict(n=len(xp), with_a_box=int(sum(len(mx_.predict(str(p), imgsz=IMGSZ, conf=thr, iou=0.5, verbose=False, device=0 if torch.cuda.is_available() else "cpu")[0].boxes) > 0 for p in xp))) for thr in FROZEN_THRESHOLDS}
        ind.update(seed=best_seed, protocol="frozen: best validation seed, thresholds 0.26 and 0.35, scored once", dev_test_mAP50_mean=float(test_df.test_mAP50.mean()))
        print(json.dumps(ind, indent=2)); (LOG_DIR / "independent_result.json").write_text(json.dumps(ind, indent=2))
        if not SMOKE and not ALLOW_RESCORE: lock.write_text(json.dumps(ind, indent=2)); print("LOCKED: the independent set is now scored; do not score it again.")
        indep_result = ind
else:
    print("no independent set for this condition (acne only) or the folder is missing")

independent set: 41 photos with boxes | 2 marked X (no acne)
{
  "mAP50": 0.35842929779085353,
  "mAP50_95": 0.11197952681999657,
  "n_photos": 41,
  "n_boxes": 437,
  "bootstrap_mAP50_95pct_range": [
    0.30616674237948194,
    0.4352194681268827
  ],
  "at_threshold": {
    "0.26": {
      "precision": 0.38422818791946306,
      "recall": 0.5240274599542334,
      "F1": 0.4433688286544046,
      "F1_95pct_range": [
        0.38308896229133876,
        0.501993722390142
      ],
      "photo_sensitivity": 1.0
    },
    "0.35": {
      "precision": 0.40963855421686746,
      "recall": 0.4668192219679634,
      "F1": 0.4363636363636363,
      "F1_95pct_range": [
        0.3741564848156859,
        0.49906719770018565
      ],
      "photo_sensitivity": 1.0
    }
  },
  "X_photos": {
    "0.26": {
      "n": 2,
      "with_a_box": 2
    },
    "0.35": {
      "n": 2,
      "with_a_box": 2
    }
  },
  "seed": 123,
  "protocol": "frozen: best validation seed, thresholds 0.26 and 0.35, s

In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 - ONNX Export, Keras/ONNX-style Verification and CPU Latency
# ─────────────────────────────────────────────────────────────────────────────
import onnxruntime as ort
best_w = seed_runs[best_seed]["best"]
onnx_path = Path(YOLO(str(best_w)).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False))
dst = EXPORT_DIR / ("lesion_detector_%s.onnx" % CONDITION); shutil.copy2(onnx_path, dst)
sha = hashlib.sha256(dst.read_bytes()).hexdigest()
sess = ort.InferenceSession(str(dst), providers=["CPUExecutionProvider"], sess_options=ort.SessionOptions())
inp, outp = sess.get_inputs()[0], sess.get_outputs()[0]
print("ONNX input", inp.shape, "| output", outp.shape, "| size %.1f MB | sha256 %s" % (dst.stat().st_size / 1e6, sha[:16]))
assert list(outp.shape)[1] == 5, "a one-class detector must output 4 box values + 1 class score (rows = 5)"
# PyTorch vs ONNX on real test photos: same boxes at the deployment threshold?
mpt = YOLO(str(best_w)); agree = []
for p in imgs_of(TEST)[:12]:
    a = mpt.predict(str(p), imgsz=IMGSZ, conf=deploy_conf, iou=0.45, verbose=False, device="cpu")[0].boxes
    mo = YOLO(str(dst), task="detect").predict(str(p), imgsz=IMGSZ, conf=deploy_conf, iou=0.45, verbose=False, device="cpu")[0].boxes
    agree.append((len(a), len(mo)))
d = [abs(x - y) for x, y in agree]; print("box counts PyTorch vs ONNX on 12 photos:", agree, "| max difference", max(d))
assert max(d) <= 2, "ONNX and PyTorch disagree by more than 2 boxes on a photo"
x = np.random.default_rng(0).random((1, 3, IMGSZ, IMGSZ), dtype=np.float32); sess.run(None, {inp.name: x})
ts = []
for _ in range(15):
    t0 = time.perf_counter(); sess.run(None, {inp.name: x}); ts.append((time.perf_counter() - t0) * 1000)
latency_ms = float(np.median(ts[3:])); print("CPU latency on THIS machine (onnxruntime, default threads): median %.0f ms per frame at %d px (measure again on the mini PC with benchmark_pc.py)" % (latency_ms, IMGSZ))

Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.14.1+cu130 CPU (13th Gen Intel Core i5-13500HX)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 44.4 GFLOPs

PyTorch: starting from '/home/bautista/PrecisionSkinV1/runs/lesion_acne_boxed_only/models/seed_123/weights/best.pt' with input shape (1, 3, 800, 800) BCHW and output shape(s) (1, 5, 13125) (21.5 MB)

ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.97...
ONNX: export success ✅ 2.0s, saved as '/home/bautista/PrecisionSkinV1/runs/lesion_acne_boxed_only/models/seed_123/weights/best.onnx' (42.8 MB)

Export complete (2.5s)
Results saved to /home/bautista/PrecisionSkinV1/runs/lesion_acne_boxed_only/models/seed_123/weights/best.onnx
Predict:         yolo predict task=detect model=/home/bautista/PrecisionSkinV1/runs/lesion_acne_boxed_only/models/seed_123/weights/best.onnx imgsz=800 
Validate:        yolo val task=detect model=/home/bautista/PrecisionSkinV1/runs/lesion_acne_boxed_only/mo

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 16 - Master Manifest, SHA-256 and Model Metadata (+ app json)
# ─────────────────────────────────────────────────────────────────────────────
meta = dict(model="PrecisionSkin %s specialist detector" % CONDITION, condition=CONDITION, classes=[CONDITION], imgsz=IMGSZ, default_conf=deploy_conf, nms_iou=0.45,
            onnx_sha256=sha, onnx_file=dst.name, source_weights=str(best_w), best_seed=best_seed, seeds=SEEDS, run_name=RUN_NAME, use_negatives=USE_NEGATIVES,
            base_model=MODEL, epochs_max=EPOCHS, dataset=str(src), dataset_fingerprint=dataset_fingerprint, primary_test=CFG["primary_test"],
            test_mean=dict(mAP50=float(test_df.test_mAP50.mean()), mAP50_95=float(test_df.test_mAP50_95.mean()), precision=float(test_df.precision.mean()), recall=float(test_df.recall.mean()), F1=float(test_df.F1.mean())),
            test_std_mAP50=float(test_df.test_mAP50.std(ddof=1)) if len(test_df) > 1 else 0.0, bootstrap_ci95_mAP50=[float(lo), float(hi)], leakfree=leakfree,
            v1_comparison=v1, cpu_latency_ms_this_machine=latency_ms, versions=dict(torch=torch.__version__, ultralytics=ultralytics.__version__), created=time.strftime("%Y-%m-%dT%H:%M:%S"))
(EXPORT_DIR / ("lesion_detector_%s.json" % CONDITION)).write_text(json.dumps(meta, indent=2))
(LOG_DIR / "model_metadata.json").write_text(json.dumps(meta, indent=2))
print("written:", EXPORT_DIR / ("lesion_detector_%s.json" % CONDITION))

written: /home/bautista/PrecisionSkinV1/runs/lesion_acne_boxed_only/export/lesion_detector_acne.json


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 17 - Install in the app (optional; set INSTALL_IN_APP = True in cell 3)
# ─────────────────────────────────────────────────────────────────────────────
if INSTALL_IN_APP and not SMOKE:
    APP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(dst, APP_MODEL_DIR / dst.name); shutil.copy2(EXPORT_DIR / ("lesion_detector_%s.json" % CONDITION), APP_MODEL_DIR / ("lesion_detector_%s.json" % CONDITION))
    print("installed in the app:", APP_MODEL_DIR / dst.name, "\nrebuild the app, then set the %s threshold in Settings (suggested %.3f)" % (CONDITION, deploy_conf))
else:
    print("not installed (INSTALL_IN_APP is False). Files are in:", EXPORT_DIR)

not installed (INSTALL_IN_APP is False). Files are in: /home/bautista/PrecisionSkinV1/runs/lesion_acne_boxed_only/export


In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 18 - Final Sanity Assertions
# ─────────────────────────────────────────────────────────────────────────────
checks = []
def chk(name, ok): checks.append((name, bool(ok))); print(("PASS " if ok else "FAIL ") + name)
chk("every seed finished (DONE marker + best.pt)", all((r["run"] / "DONE").exists() and r["best"].exists() for r in seed_runs.values()))
chk("test photos are not in train/val (file names)", not any(names["train"] & names[t] or names["val"] & names[t] for t in CFG["test_dirs"]))
chk("best seed was chosen on validation only (cell 9 ran before cell 10)", best_seed in set(val_df.seed))
chk("validation and test sets are non-empty", len(imgs_of(DATA_LOCAL / "val")) > 0 and len(imgs_of(TEST)) > 0)
chk("negatives only if USE_NEGATIVES", (info["negatives"] > 0) == USE_NEGATIVES or SMOKE)
chk("independent set scored at most once (lock file) or not applicable", SMOKE or CONDITION != "acne" or not INDEPENDENT_DIR.exists() or (INDEPENDENT_DIR / "SCORED.json").exists() or indep_result is None)
chk("ONNX file matches its recorded SHA-256", hashlib.sha256(dst.read_bytes()).hexdigest() == meta["onnx_sha256"])
chk("ONNX is single-class (5 output rows)", list(outp.shape)[1] == 5)
chk("metadata json has classes / imgsz / default_conf / nms_iou (the app reads these)", all(k in meta for k in ("classes", "imgsz", "default_conf", "nms_iou")))
chk("the app's class list matches the condition", meta["classes"] == [CONDITION])
assert all(ok for _, ok in checks), "a sanity check failed - do not deploy"
print("\nall checks passed")

PASS every seed finished (DONE marker + best.pt)
PASS test photos are not in train/val (file names)
PASS best seed was chosen on validation only (cell 9 ran before cell 10)
PASS validation and test sets are non-empty
PASS negatives only if USE_NEGATIVES
PASS independent set scored at most once (lock file) or not applicable
PASS ONNX file matches its recorded SHA-256
PASS ONNX is single-class (5 output rows)
PASS metadata json has classes / imgsz / default_conf / nms_iou (the app reads these)
PASS the app's class list matches the condition

all checks passed


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 19 - Deployment Summary Export
# ─────────────────────────────────────────────────────────────────────────────
m = meta["test_mean"]
txt = ["# Lesion detector - deployment summary (%s, run %s)" % (CONDITION, RUN_NAME), "",
       "Dataset: `%s` | fingerprint %s" % (src, dataset_fingerprint[:16]), manifest.to_string(index=False), "",
       "Model: %s, %d px, seeds %s (best on validation: %d), negatives: %s" % (MODEL, IMGSZ, SEEDS, best_seed, USE_NEGATIVES), "",
       "Test (%s, %d photos, never used for selection), mean over seeds: mAP@0.5 %.3f (std %.3f, 95%% range %.3f-%.3f) | mAP@0.5-0.95 %.3f | precision %.3f | recall %.3f | F1 %.3f" % (
           CFG["primary_test"], n, m["mAP50"], meta["test_std_mAP50"], lo, hi, m["mAP50_95"], m["precision"], m["recall"], m["F1"]),
       "Leak-free re-evaluation: %d look-alike test photos removed -> mAP@0.5 %.3f" % (leakfree["n_removed"], leakfree["test_mAP50"])]
if indep_result: txt.append("INDEPENDENT set (%d photos, %d boxes, scored once): mAP@0.5 %.3f (95%% range %.3f-%.3f) | at 0.26: P %.2f R %.2f F1 %.2f | at 0.35: P %.2f R %.2f F1 %.2f" % (indep_result["n_photos"], indep_result["n_boxes"], indep_result["mAP50"], indep_result["bootstrap_mAP50_95pct_range"][0], indep_result["bootstrap_mAP50_95pct_range"][1], indep_result["at_threshold"]["0.26"]["precision"], indep_result["at_threshold"]["0.26"]["recall"], indep_result["at_threshold"]["0.26"]["F1"], indep_result["at_threshold"]["0.35"]["precision"], indep_result["at_threshold"]["0.35"]["recall"], indep_result["at_threshold"]["0.35"]["F1"]))
if v1: txt.append("v1 (in the app): mAP@0.5 %.3f, precision %.3f, recall %.3f, F1 %.3f" % (v1["test_mAP50"], v1["precision"], v1["recall"], v1["F1"]))
txt += ["", "Deployment: confidence %.3f (validation); app default %.3f; ONNX sha256 `%s`; CPU latency on the training machine %.0f ms/frame (re-measure on the mini PC)." % (deploy_conf, CFG["app_threshold"], sha, latency_ms),
        "", "Limits: single hand-boxed test set; boxes are candidate lesions, not a diagnosis; tiny spots are rarely found (see cell 13)."]
(RUN_ROOT / "DEPLOYMENT_SUMMARY.md").write_text("\n".join(txt), encoding="utf-8"); print("\n".join(txt))

# Lesion detector - deployment summary (acne, run boxed_only)

Dataset: `/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Acne_Boxing/acne_dataset_v4` | fingerprint ae51483a7b72c113
split  photos  boxes  boxes_per_photo  photos_without_boxes
train    1145  11621             10.1                     0
  val     154   1562             10.1                     0
 test     167   1369              8.2                     0

Model: yolov8s.pt, 800 px, seeds [42, 7, 123] (best on validation: 123), negatives: False

Test (test, 167 photos, never used for selection), mean over seeds: mAP@0.5 0.413 (std 0.006, 95% range 0.369-0.463) | mAP@0.5-0.95 0.123 | precision 0.427 | recall 0.543 | F1 0.478
Leak-free re-evaluation: 2 look-alike test photos removed -> mAP@0.5 0.418
INDEPENDENT set (41 photos, 437 boxes, scored once): mAP@0.5 0.358 (95% range 0.306-0.435) | at 0.26: P 0.38 R 0.52 F1 0.44 | at 0.35: P 0.41 R 0.47 F1 0.44
v1 (in the app): mAP@0.5 0.220, precision 0.359, recall 0.234, F1 0.283

De